# 03 - Ball values
Every ball gets a value = what happened minus what an average player does in that exact state (out-of-fold expectation).
Components are kept separate because they have very different reliability:
  vr = runs vs expectation, vo = wicket vs expectation (priced at the state-specific run cost of a wicket),
  batters: vdot = dot balls avoided vs expectation, vbnd = boundaries vs expectation
  bowlers: vdot = dot balls bowled vs expectation, vext = extras avoided vs expectation.
Expectations now also know about dot streaks, rising required rate, free hits and the bowler's over so far.
Extras are charged in full (wide = runs conceded, no-ball = runs + the free-hit premium): the old version scored a wide as a cheap delivery.
Outputs: outputs/batter_ball_values.parquet, outputs/bowler_ball_values.parquet

In [1]:
import warnings
import numpy as np, pandas as pd
from pathlib import Path
from sklearn.ensemble import HistGradientBoostingRegressor as HGR, HistGradientBoostingClassifier as HGC
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_rows', 80)
ROOT = Path.cwd() if (Path.cwd() / 'data').exists() else Path.cwd().parent
DATA, OUT = ROOT / 'data', ROOT / 'outputs'
b = pd.read_parquet(OUT / 'balls_pressure.parquet')

TIER_W = {'IPL': 1.0, 'T20I_FM': 1.0, 'T20I_ASSOC': 0.5, 'A_TEAM': 0.5}   # evidence weight by opposition quality

## 1. Expected outcome for an average player in the same state (out-of-fold, grouped by match)
Evaluation set: regulation innings of completed matches. No-result and DLS matches are excluded (state is unreliable).
Super overs are kept and valued against last-over expectations.

In [2]:
b['ps_c'] = b.pace_or_spin.map({'Pace': 0, 'Spin': 1})
b['bo_c'] = np.select([b.batting_order <= 2, b.batting_order <= 4, b.batting_order <= 6], [0, 1, 2], 3)
BASE = ['innings', 'overid', 'wkts', 'balls_left', 'crr', 'rrr', 'runs_needed', 'bat_balls_before', 'g_c', 'bt_c', 'ot_c', 'ps_c', 'bo_c']
FE_BAT = BASE + ['dot_streak', 'dots_last6', 'rrr_trend', 'free_hit']
FE_BOWL = BASE + ['bowler_dots_over', 'bowler_extras_over', 'free_hit']
CATN = ['g_c', 'bt_c', 'ot_c', 'ps_c', 'bo_c']

E = b[b.innings.isin([1, 2]) & b.status.isin(['ok', 'tied']) & ~b.dls & (b.balls_left >= 1)].copy()
perm = np.random.default_rng(7).permutation(E.match_id.unique())
E['fold'] = E.match_id.map({m: i % 5 for i, m in enumerate(perm)})


def oof(y, mask, kind, feats):
    cat = [feats.index(c) for c in CATN]
    pred = pd.Series(np.nan, index=E.index)
    for f in range(5):
        tr, te = E[mask & (E.fold != f)], E[mask & (E.fold == f)]
        M = HGC if kind == 'c' else HGR
        mdl = M(max_iter=200, learning_rate=0.06, max_leaf_nodes=24, min_samples_leaf=200, l2_regularization=1.0,
                categorical_features=cat, random_state=0).fit(tr[feats], y[tr.index])
        pred[te.index] = mdl.predict_proba(te[feats])[:, 1] if kind == 'c' else mdl.predict(te[feats])
    return pred


allrows = pd.Series(True, index=E.index)
fb = E.faced
E['exp_runs'] = oof(E.runs, fb, 'r', FE_BAT)
E['exp_out'] = oof(E.batter_out.astype(int), fb, 'c', FE_BAT)
E['exp_dot'] = oof(E.bdot.astype(int), fb, 'c', FE_BAT)
E['exp_bnd'] = oof(E.is_bnd.astype(int), fb, 'c', FE_BAT)
E['exp_bruns'] = oof(E.bowler_runs, E.legal, 'r', FE_BOWL)           # legal balls only: extras are charged separately
E['exp_bdot'] = oof(E.bowl_dot.astype(int), E.legal, 'c', FE_BOWL)
E['exp_bwkt'] = oof(E.bowler_wkt.astype(int), allrows, 'c', FE_BOWL)
E['exp_ext'] = oof(E.bowl_extra.astype(int), allrows, 'c', FE_BOWL)

# a no-ball hands the batter a free hit: premium (runs) on the next legal ball, charged to the bowler
fh = E[E.legal]
FH_PREM = fh[fh.free_hit == 1].bowler_runs.mean() - fh[fh.free_hit == 0].bowler_runs.mean()
print('free-hit premium charged to a no-ball (runs): %.2f' % FH_PREM)

# super overs: expectation = regulation last-over averages
last = E[E.overid == 20]
SO = b[b.is_super_over].copy()
for c, src in [('exp_runs', last[last.faced].runs), ('exp_out', last[last.faced].batter_out), ('exp_dot', last[last.faced].bdot),
               ('exp_bnd', last[last.faced].is_bnd), ('exp_bruns', last[last.legal].bowler_runs), ('exp_bdot', last[last.legal].bowl_dot),
               ('exp_bwkt', last.bowler_wkt), ('exp_ext', last.bowl_extra)]:
    SO[c] = src.astype(float).mean()
SO['wkt_cost'] = last.wkt_cost.mean()
D = pd.concat([E, SO], ignore_index=True)
D['w'] = D.match_tier.map(TIER_W)
print('wicket cost (runs) by phase:', D.groupby('phase').wkt_cost.mean().round(1).to_dict())
print('OOF check, bat runs/ball %.3f vs %.3f | bat dot rate %.3f vs %.3f | boundary rate %.3f vs %.3f' % (
    E[E.faced].runs.mean(), E[E.faced].exp_runs.mean(), E[E.faced].bdot.mean(), E[E.faced].exp_dot.mean(), E[E.faced].is_bnd.mean(), E[E.faced].exp_bnd.mean()))
print('OOF check, bowler dot rate %.3f vs %.3f | extras rate %.3f vs %.3f' % (
    E[E.legal].bowl_dot.mean(), E[E.legal].exp_bdot.mean(), E.bowl_extra.mean(), E.exp_ext.mean()))

free-hit premium charged to a no-ball (runs): 0.93
wicket cost (runs) by phase: {'DEATH': 2.5, 'MID': 5.7, 'PP': 9.8, 'SO': 1.4}
OOF check, bat runs/ball 1.393 vs 1.394 | bat dot rate 0.371 vs 0.371 | boundary rate 0.186 vs 0.187
OOF check, bowler dot rate 0.371 vs 0.371 | extras rate 0.043 vs 0.043


## 2. Ball values (runs-equivalent for runs / wickets; probability units for the style components)

In [3]:
Bt = D[D.faced].copy()
Bt['vr'] = Bt.runs - Bt.exp_runs
Bt['vo'] = -(Bt.batter_out.astype(float) - Bt.exp_out) * Bt.wkt_cost
Bt['vdot'] = -(Bt.bdot.astype(float) - Bt.exp_dot)           # positive = fewer dots than expected
Bt['vbnd'] = Bt.is_bnd.astype(float) - Bt.exp_bnd            # positive = more boundaries than expected
Bt['v'] = Bt.vr + Bt.vo

Bw = D.copy()
nb = Bw.extras_type.eq('Overstep')
Bw['vr'] = np.where(Bw.legal, -(Bw.bowler_runs - Bw.exp_bruns), -(Bw.bowler_runs + FH_PREM * nb))   # extras charged in full
Bw['vo'] = (Bw.bowler_wkt.astype(float) - Bw.exp_bwkt) * Bw.wkt_cost
Bw['vdot'] = np.where(Bw.legal, Bw.bowl_dot.astype(float) - Bw.exp_bdot, np.nan)                    # positive = more dots than expected
Bw['vext'] = -(Bw.bowl_extra.astype(float) - Bw.exp_ext)                                            # positive = fewer extras than expected
Bw['v'] = Bw.vr + Bw.vo

COMMON = ['match_id', 'innings', 'innings_ball_num', 'season', 'comp_group', 'match_tier', 'match_date', 'phase', 'overid',
          'striker_id', 'bowler_id', 'bat_team', 'bowl_team', 'runs', 'tot', 'legal', 'faced', 'batter_out', 'bowler_runs',
          'bowler_wkt', 'is_four', 'is_six', 'is_bnd', 'bdot', 'bowl_dot', 'bowl_extra', 'extras_type', 'is_super_over',
          'p_bat', 'p_bowl', 'w', 'wp', 'rrr', 'crr', 'rrr_trend', 'dot_streak', 'free_hit', 'v', 'vr', 'vo', 'vdot']
Bt[COMMON + ['vbnd', 'exp_runs', 'exp_dot', 'exp_bnd']].to_parquet(OUT / 'batter_ball_values.parquet')
Bw[COMMON + ['vext', 'exp_bruns', 'exp_bdot', 'exp_ext']].to_parquet(OUT / 'bowler_ball_values.parquet')
print('batter ball SD: runs-part %.2f, wicket-part %.2f' % (Bt.vr.std(), Bt.vo.std()))
print('bowler ball SD: runs-part %.2f, wicket-part %.2f' % (Bw.vr.std(), Bw.vo.std()))
x = Bw[~Bw.legal & Bw.p_bowl.ge(80)]
print('bowler value on extras delivered under pressure: mean %.2f runs (old version: +0.30, i.e. rewarded)' % x.vr.mean())
print('saved ball values:', len(Bt), 'batter rows,', len(Bw), 'bowler rows')

batter ball SD: runs-part 1.71, wicket-part 1.64
bowler ball SD: runs-part 1.71, wicket-part 1.60
bowler value on extras delivered under pressure: mean -1.37 runs (old version: +0.30, i.e. rewarded)
saved ball values: 188097 batter rows, 195619 bowler rows
